# Job B — Feature-based Deceuninck benchmark (Colab)

Runs PatchCore + PaDiM + SubspaceAD on the Deceuninck industrial dataset
with the **val_defect calibration rules**:

- `thresholding.mode: val_f1` — threshold picked at the F1 maximum on a val
  split that contains both classes (instead of the 99th-percentile-of-clean
  fallback).
- The patched splitter routes 10% of anomaly images into val while keeping
  the training split good-only, so calibration changes without changing the
  one-class fitting setup.

Per-model summary now also includes `recall_at_fpr_1pct`,
`recall_at_fpr_5pct`, `macro_recall`, `weighted_recall` and a
`per_defect_recall` dict — see [METHOD.md §4](../METHOD.md).

**Before running:**
- Runtime -> Change runtime type -> GPU (T4/V100/A100).
- `Deceuninck_dataset/{good,defects}/` must be on Drive (unzipped, BMPs).

**Outputs:** synced to `/content/drive/MyDrive/thesis_runs/jobB_val_defect/`.
Safe to interrupt and restart — a `jobB_val_defect.done` marker is written
on success and subsequent runs exit early until the marker is deleted.

In [ ]:
# 1. Mount Drive and verify the Deceuninck folders are visible.
# Deceuninck stores BMPs (line-camera native format), not JPEGs.
from google.colab import drive
drive.mount('/content/drive')

import os, glob
DATASET_DIR = '/content/drive/MyDrive/Deceuninck_dataset'
good_imgs    = glob.glob(os.path.join(DATASET_DIR, 'good', '*.bmp'))
defect_dirs  = sorted(d for d in glob.glob(os.path.join(DATASET_DIR, 'defects', '*')) if os.path.isdir(d))
defect_imgs  = sum(len(glob.glob(os.path.join(d, '*.bmp'))) for d in defect_dirs)
print(f'good/    : {len(good_imgs)} images')
print(f'defects/ : {len(defect_dirs)} subfolders, {defect_imgs} images total')
for d in defect_dirs:
    n = len(glob.glob(os.path.join(d, '*.bmp')))
    print(f'  - {os.path.basename(d):40s} {n} images')
assert len(good_imgs) > 0 and defect_imgs > 0, 'Dataset empty — check DATASET_DIR matches your Drive layout.'

In [ ]:
# 2. Clone (or pull) the thesis repo into /content/.
# Replace <your-username>/<your-repo> with the repo you pushed to.
REPO_URL = 'https://github.com/LorenzSF/Real-time-visual-defect-detection.git'
REPO_DIR = '/content/Real-time-visual-defect-detection'
BRANCH   = 'main'

import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', '--all'])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])

print('Repo ready at', REPO_DIR)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 3. Install Python dependencies for the three models.
# Same pins as Job A — kept in sync so results are directly comparable.
!pip install -q "anomalib>=2.2,<3" lightning "transformers>=4.46,<5" scikit-learn timm kornia einops
!pip install -q -e /content/Real-time-visual-defect-detection --no-deps || echo '[note] editable install skipped (not required)'

import torch, anomalib, lightning, transformers
print('torch       ', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print('anomalib    ', anomalib.__version__)
print('lightning   ', lightning.__version__)
print('transformers', transformers.__version__)

In [ ]:
# 4. Launch Job B (patched splitter + val_f1). Streams logs live; safe to interrupt and rerun.
import os
os.environ['DATASET_DIR'] = '/content/drive/MyDrive/Deceuninck_dataset'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/thesis_runs/jobB_val_defect'
os.environ['WORK_DIR']    = '/content/work'
os.environ['REPO_DIR']    = '/content/Real-time-visual-defect-detection'
os.environ['CONFIG']      = '/content/Real-time-visual-defect-detection/src/benchmark_AD/configs/colab_featurebased_deceuninck_val_defect.yaml'

# Ensure the Drive log dir exists before `tee` tries to append to it.
!mkdir -p {os.environ['RESULTS_DIR']}
!bash /content/Real-time-visual-defect-detection/scripts/run_jobB_val_defect_colab.sh 2>&1 | tee -a {os.environ['RESULTS_DIR']}/run.log

## Troubleshooting

- **`Expected good/ and defects/ under ...`**: the Drive folder doesn't match the expected layout. Confirm `ls /content/drive/MyDrive/Deceuninck_dataset` shows `good/` and `defects/` at the top level.
- **`good/    : 0 images`**: the loader scans `*.bmp`. If your copy on Drive is JPEG, either re-export the dataset as BMP or relax the glob in cell 1 (the pipeline itself accepts `.bmp`, `.jpg`, `.jpeg`, `.png`, `.tif`, `.tiff` — see `SUPPORTED_EXTS` in [src/benchmark_AD/data.py](../src/benchmark_AD/data.py)).
- **Drive copy is slow**: Deceuninck is many small BMPs and Drive throttles per-file I/O. If the copy stalls, consider zipping the folder once on Drive (`Deceuninck_dataset.zip`) and adapting the driver to `unzip -q` on the local SSD like Job A does.
- **`CUDA out of memory` on SubspaceAD**: lower `subspacead.batch_size` in [colab_featurebased_deceuninck_val_defect.yaml](../src/benchmark_AD/configs/colab_featurebased_deceuninck_val_defect.yaml) (default 4) to 2.
- **Session disconnected mid-run**: reopen the notebook, re-run cells 1-3, then rerun cell 4. Delete `/content/drive/MyDrive/thesis_runs/jobB_val_defect/jobB_val_defect.done` first if you want to restart the pipeline itself.
- **Results look off**: check `run.log` in `RESULTS_DIR` — the pipeline prints per-model AUROC/F1 and any skipped-model warnings there. The per-defect breakdown is in `benchmark_summary.json` under each model's `per_defect_recall` field.

In [ ]:
# 6. Release Colab resources once the run is finished and synced.
# This clears local caches first, then disconnects the runtime.
import gc

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

gc.collect()

try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    pass
